# Feature Engineering

## 1. Import Libraries

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import networkx as nx
from itertools import combinations
from joblib import Parallel, delayed
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Load Processed Data

In [ ]:
PATH_PREFIX = '../'

data_tran = pd.read_json(PATH_PREFIX + 'data/data_processed/data_tran.json', orient='index')
data_test = pd.read_json(PATH_PREFIX + 'data/data_processed/data_test.json', orient='index')

In [ ]:
n_tran = data_tran.shape[0]
n_test = data_test.shape[0]

In [ ]:
data_tran.head(3)

In [ ]:
data_test.head(3)

## 3. Constract the Historical-Reproduction Based Features

### 3.1 Feature Engineering on Co-authorship Network 

In [ ]:
def get_coauthors_graph(data):

    coauthors_graph = nx.Graph()

    for authors in data['authors']:
        for author_pair in combinations(authors, 2):
            if coauthors_graph.has_edge(*author_pair):
                coauthors_graph[author_pair[0]][author_pair[1]]['weight'] += 1
            else:
                coauthors_graph.add_edge(author_pair[0], author_pair[1], weight=1)
    
    return coauthors_graph

coauthors_graph = get_coauthors_graph(data_tran)

In [ ]:
def get_coauthors_vector(graph, start_nodes):

    node_array = np.zeros((1, 100)) 

    def dfs_iterative(graph, start_node):
        
        if start_node not in graph:
            return
        
        stack = [(start_node, 1)] 
        visited = set() 

        while stack:
            
            node, depth = stack.pop()

            if node in visited:
                continue 

            visited.add(node)

            for neighbor, edge in graph[node].items():
                weight = edge['weight'] * (1 / (depth * math.log(depth + 1)))
                if 0 <= neighbor < 100: 
                    node_array[0, neighbor] += weight 
                if neighbor not in visited:
                    stack.append((neighbor, depth + 1)) 

    for start_node in start_nodes:
        dfs_iterative(graph, start_node)

    return node_array.reshape(1, 100)

In [ ]:
def get_coauthors_matrix(data, graph):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_coauthors_vector)(graph, row['authors_coop']) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [ ]:
if os.path.exists(PATH_PREFIX + 'data/data_feature/x_tran_coauthors_m1.npy'):
    x_tran_coauthors = np.load(PATH_PREFIX + 'data/data_feature/x_tran_coauthors_m1.npy')
else:
    x_tran_coauthors = get_coauthors_matrix(data_tran, coauthors_graph)
    np.save(PATH_PREFIX + 'data/data_feature/x_tran_coauthors_m1.npy', x_tran_coauthors)

if os.path.exists(PATH_PREFIX + 'data/data_feature/x_test_coauthors_m1.npy'):
    x_test_coauthors = np.load(PATH_PREFIX + 'data/data_feature/x_test_coauthors_m1.npy')
else:
    x_test_coauthors = get_coauthors_matrix(data_test, coauthors_graph)
    np.save(PATH_PREFIX + 'data/data_feature/x_test_coauthors_m1.npy', x_test_coauthors)

### 3.2 Feature Engineering on Publication Venues

#### 3.2.1 Feature Engineering on Publication Venues based on Venue Histories

In [ ]:
def get_venue_dict_a(data):
    
    num_venues=466
    vector_size=21246

    venue_dict = {venue: np.zeros(vector_size, dtype=int) for venue in range(num_venues)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        venue = row['venue'] 
        authors = row['authors'] 

        for author_id in authors:
            if author_id < 21246: 
                venue_dict[venue][author_id] += 1

    return venue_dict

venue_dict_a = get_venue_dict_a(data_tran)

In [ ]:
def get_venue_array_a(venue, venue_dict):
    return np.array(venue_dict[venue][:100]).reshape(1, 100)

In [ ]:
def get_venue_matrix_a(data, venue_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_venue_array_a)(row['venue'], venue_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [ ]:
if os.path.exists(PATH_PREFIX + 'data/data_feature/x_tran_venue_a_m1.npy'):
    x_tran_venue_a = np.load(PATH_PREFIX + 'data/data_feature/x_tran_venue_a_m1.npy')
else:
    x_tran_venue_a = get_venue_matrix_a(data_tran, venue_dict_a)
    np.save(PATH_PREFIX + 'data/data_feature/x_tran_venue_a_m1.npy', x_tran_venue_a)

if os.path.exists(PATH_PREFIX + 'data/data_feature/x_test_venue_a_m1.npy'):
    x_test_venue_a = np.load(PATH_PREFIX + 'data/data_feature/x_test_venue_a_m1.npy')
else:
    x_test_venue_a = get_venue_matrix_a(data_test, venue_dict_a)
    np.save(PATH_PREFIX + 'data/data_feature/x_test_venue_a_m1.npy', x_test_venue_a)

#### 3.2.2 Feature Engineering on Publication Venues based on Venue Histories & Coauthorship Network

In [ ]:
def get_venue_dict_b(data):
    
    num_venues=466
    vector_size=21246

    venue_dict = {venue: np.zeros(vector_size, dtype=int) for venue in range(num_venues)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        venue = row['venue'] 
        authors = row['authors'] 

        for author_id in authors:
            if author_id < 21246: 
                venue_dict[venue][author_id] += 1

    return venue_dict

venue_dict_b = get_venue_dict_b(data_tran)

In [ ]:
def get_venue_vector_b(coauthor_list, venue_dict):

    temp_array = np.zeros(100) 

    for author in coauthor_list:
        relevant_venues = [venue for venue, array in venue_dict.items() if array[author] > 0]
        for venue in relevant_venues:
            temp_array += venue_dict[venue][:100]

    return temp_array.reshape(1, 100)

In [ ]:
def get_venue_matrix_b(data, venue_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_venue_vector_b)(row['authors_coop'], venue_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [ ]:
if os.path.exists(PATH_PREFIX + 'data/data_feature/x_tran_venue_b_m1.npy'):
    x_tran_venue_b = np.load(PATH_PREFIX + 'data/data_feature/x_tran_venue_b_m1.npy')
else:
    x_tran_venue_b = get_venue_matrix_b(data_tran, venue_dict_b)
    np.save(PATH_PREFIX + 'data/data_feature/x_tran_venue_b_m1.npy', x_tran_venue_b)

if os.path.exists(PATH_PREFIX + 'data/data_feature/x_test_venue_b_m1.npy'):
    x_test_venue_b = np.load(PATH_PREFIX + 'data/data_feature/x_test_venue_b_m1.npy')
else:
    x_test_venue_b = get_venue_matrix_b(data_test, venue_dict_b)
    np.save(PATH_PREFIX + 'data/data_feature/x_test_venue_b_m1.npy', x_test_venue_b)

### 3.3 Feature Engineering on Text

#### 3.3.1 Feature Engineering on Text Histories

In [ ]:
tfidf_encoder_a = TfidfVectorizer(ngram_range=(1, 2))
tfidf_encoder_a.fit(data_tran['str_combined']) 

def get_top_tfidf_words_a(text, top_n):
    tfidf_array = tfidf_encoder_a.transform([text]).toarray()[0]
    top_indices = np.argsort(tfidf_array)[-top_n:][::-1] 
    return top_indices

In [ ]:
def get_word_author_dict_a(data, top_n):

    num_authors = 21246
    word_author_dict = {author_id: {} for author_id in range(num_authors)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        text = row['str_combined'] 
        authors = row['authors'] 

        top_words = get_top_tfidf_words_a(text, top_n) 

        for author in authors:
            if author >= 0: 
                for word_id in top_words:
                    if word_id in word_author_dict[author]:
                        word_author_dict[author][word_id] += 1
                    else:
                        word_author_dict[author][word_id] = 1

    return word_author_dict

word_author_dict_a = get_word_author_dict_a(data_tran, top_n=10)

In [ ]:
def get_text_vector_a(text, word_author_dict):

    result_array = np.zeros(100) 
    top_words = get_top_tfidf_words_a(text, top_n=20) 

    for main_author in range(100):
        if main_author not in word_author_dict:
            continue 

        for word in top_words:
            if word in word_author_dict[main_author]:
                result_array[main_author] += word_author_dict[main_author][word]

    return result_array.reshape(1, 100)

In [ ]:
def get_text_matrix_a(data, word_author_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_text_vector_a)(row['str_combined'], word_author_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [ ]:
if os.path.exists(PATH_PREFIX + 'data/data_feature/x_tran_text_a_m1.npy'):
    x_tran_text_a = np.load(PATH_PREFIX + 'data/data_feature/x_tran_text_a_m1.npy')
else:
    x_tran_text_a = get_text_matrix_a(data_tran, word_author_dict_a)
    np.save(PATH_PREFIX + 'data/data_feature/x_tran_text_a_m1.npy', x_tran_text_a)

if os.path.exists(PATH_PREFIX + 'data/data_feature/x_test_text_a_m1.npy'):
    x_test_text_a = np.load(PATH_PREFIX + 'data/data_feature/x_test_text_a_m1.npy')
else:
    x_test_text_a = get_text_matrix_a(data_test, word_author_dict_a)
    np.save(PATH_PREFIX + 'data/data_feature/x_test_text_a_m1.npy', x_test_text_a)

#### 3.3.2 Feature Engineering on Text Histories & Coauthorship Network

In [ ]:
tfidf_encoder_b = TfidfVectorizer(ngram_range=(1, 2))
tfidf_encoder_b.fit(data_tran['str_combined']) 

def get_top_tfidf_words_b(text, top_n):
    tfidf_array = tfidf_encoder_b.transform([text]).toarray()[0]
    top_indices = np.argsort(tfidf_array)[-top_n:][::-1] 
    return top_indices

In [ ]:
def get_word_author_dict_b(data, top_n):

    num_authors = 21246
    word_author_dict = {author_id: {} for author_id in range(num_authors)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        text = row['str_combined'] 
        authors = row['authors'] 

        top_words = get_top_tfidf_words_b(text, top_n) 

        for author in authors:
            if author >= 0: 
                for word_id in top_words:
                    if word_id in word_author_dict[author]:
                        word_author_dict[author][word_id] += 1
                    else:
                        word_author_dict[author][word_id] = 1

    return word_author_dict

word_author_dict_b = get_word_author_dict_b(data_tran, top_n=10)

In [ ]:
def get_text_vector_b(coauthor_list, word_author_dict):

    result_array = np.zeros(100)

    for coauthor in coauthor_list:
        if coauthor not in word_author_dict:
            continue 

        common_words = word_author_dict[coauthor].keys()
        
        for main_author in range(100):
            if main_author not in word_author_dict:
                continue 
            
            for word in common_words:
                if word in word_author_dict[main_author]:
                    result_array[main_author] += word_author_dict[main_author][word]

    return result_array.reshape(1, 100)

In [ ]:
def get_text_matrix_b(data, word_author_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_text_vector_b)(row['authors_coop'], word_author_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [ ]:
if os.path.exists(PATH_PREFIX + 'data/data_feature/x_tran_text_b_m1.npy'):
    x_tran_text_b = np.load(PATH_PREFIX + 'data/data_feature/x_tran_text_b_m1.npy')
else:
    x_tran_text_b = get_text_matrix_b(data_tran, word_author_dict_b)
    np.save(PATH_PREFIX + 'data/data_feature/x_tran_text_b_m1.npy', x_tran_text_b)

if os.path.exists(PATH_PREFIX + 'data/data_feature/x_test_text_b_m1.npy'):
    x_test_text_b = np.load(PATH_PREFIX + 'data/data_feature/x_test_text_b_m1.npy')
else:
    x_test_text_b = get_text_matrix_b(data_test, word_author_dict_b)
    np.save(PATH_PREFIX + 'data/data_feature/x_test_text_b_m1.npy', x_test_text_b)